In [1]:
# توليد بيانات رحلات فوضوية لاختبار خطوات التنظيف والجودة.
import csv
import random
from datetime import datetime, timedelta


file_path = "twenty_million_messy_trips.csv"
total_rows = 10_000_000

statuses = [
    "Completed",
    "completed",
    "COMPLETED",
    " Cancelled ",
    "failed",
    "NULL",
    "NaN",
    "Unknown"
]

cities = [
    "Khartoum",
    " Khartoum ",
    "Omdurman",
    "Bahri",
    "Port Sudan"
]

phones = [
    "0912345678",
    "00249912345678",
    "+249912345678",
    "N/A",
    "09-123-4567"
]

devices = [
    '{"os": "Android", "app_version": "5.1.2"}',
    '{"os": "iOS", "app_version": "5.1.0"}',
    '{"os": "Huawei", "app_version": "4.9.0"}',
    '{}',
    'NULL'
]

headers = [
    "trip_id",
    "driver_id",
    "customer_id",
    "start_time",
    "end_time",
    "distance_km",
    "fare_amount",
    "status",
    "phone_number",
    "device_info",
    "city"
]

print(f"⏳ بدء توليد {total_rows:,} سطر بيانات فوضوية...")

base_time = datetime(2026, 1, 1)

with open(
    file_path,
    mode="w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)
    writer.writerow(headers)

    for i in range(1, total_rows + 1):


        id_rand = random.random()

        if id_rand < 0.005:
            trip_id = ""

        elif id_rand < 0.015:
            trip_id = f"TRIP_{max(1, i - 10)}"

        else:
            trip_id = f"TRIP_{i}"


        if random.random() < 0.01:

            driver_id = "DRV_999999"
            customer_id = "CST_999999"

        else:

            driver_id = f"DRV_{random.randint(1, 1000)}"
            customer_id = f"CST_{random.randint(1, 5000)}"


        start_date = base_time + timedelta(
            seconds=i * 0.1
        )

        time_rand = random.random()

        if time_rand < 0.005:

            end_date = start_date - timedelta(
                minutes=random.randint(10, 60)
            )

        elif time_rand < 0.01:

            start_date = datetime(2030, 12, 31)

            end_date = start_date + timedelta(
                minutes=15
            )

        else:

            end_date = start_date + timedelta(
                minutes=random.randint(5, 45)
            )

        start_time_str = start_date.strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        end_time_str = end_date.strftime(
            "%Y-%m-%d %H:%M:%S"
        )


        distance = round(
            random.uniform(0.5, 500),
            2
        )

        if random.random() < 0.01:
            distance = -abs(distance)


        fare = round(
            random.uniform(50, 1500),
            2
        )

        if random.random() < 0.01:
            fare = random.choice([
                -abs(fare),
                100000,
                500000,
                1000000
            ])


        status = random.choice(statuses)
        phone = random.choice(phones)
        device_info = random.choice(devices)
        city = random.choice(cities)


        writer.writerow([
            trip_id,
            driver_id,
            customer_id,
            start_time_str,
            end_time_str,
            distance,
            fare,
            status,
            phone,
            device_info,
            city
        ])

        if i % 500_000 == 0:
            print(
                f"✅ تم إنشاء {i:,} / {total_rows:,} صف"
            )

print(
    f"✅ اكتمل توليد الملف: {file_path}"
)


⏳ بدء توليد 10,000,000 سطر بيانات فوضوية...
✅ تم إنشاء 500,000 / 10,000,000 صف
✅ تم إنشاء 1,000,000 / 10,000,000 صف
✅ تم إنشاء 1,500,000 / 10,000,000 صف
✅ تم إنشاء 2,000,000 / 10,000,000 صف
✅ تم إنشاء 2,500,000 / 10,000,000 صف
✅ تم إنشاء 3,000,000 / 10,000,000 صف
✅ تم إنشاء 3,500,000 / 10,000,000 صف
✅ تم إنشاء 4,000,000 / 10,000,000 صف
✅ تم إنشاء 4,500,000 / 10,000,000 صف
✅ تم إنشاء 5,000,000 / 10,000,000 صف
✅ تم إنشاء 5,500,000 / 10,000,000 صف
✅ تم إنشاء 6,000,000 / 10,000,000 صف
✅ تم إنشاء 6,500,000 / 10,000,000 صف
✅ تم إنشاء 7,000,000 / 10,000,000 صف
✅ تم إنشاء 7,500,000 / 10,000,000 صف
✅ تم إنشاء 8,000,000 / 10,000,000 صف
✅ تم إنشاء 8,500,000 / 10,000,000 صف
✅ تم إنشاء 9,000,000 / 10,000,000 صف
✅ تم إنشاء 9,500,000 / 10,000,000 صف
✅ تم إنشاء 10,000,000 / 10,000,000 صف
✅ اكتمل توليد الملف: twenty_million_messy_trips.csv


In [2]:
# تحميل البيانات الخام إلى DuckDB وتحويل أنواع البيانات قبل التحليل.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("DROP TABLE IF EXISTS main.my_clean_table;")

con.execute("""
CREATE TABLE main.my_clean_table AS
SELECT
    CAST(trip_id AS VARCHAR) AS trip_id,
    CAST(driver_id AS VARCHAR) AS driver_id,
    CAST(customer_id AS VARCHAR) AS customer_id,

    TRY_CAST(start_time AS TIMESTAMP) AS start_time,
    TRY_CAST(end_time AS TIMESTAMP) AS end_time,

    TRY_CAST(distance_km AS DOUBLE) AS distance_km,

    TRY_CAST(
        REPLACE(fare_amount, ',', '')
        AS DOUBLE
    ) AS fare_amount,

    CAST(status AS VARCHAR) AS status,
    CAST(phone_number AS VARCHAR) AS phone_number,
    CAST(device_info AS VARCHAR) AS device_info,
    CAST(city AS VARCHAR) AS city

FROM read_csv(
    'twenty_million_messy_trips.csv',
    HEADER = TRUE,
    ALL_VARCHAR = TRUE
);
""")

total = con.execute("""
SELECT COUNT(*)
FROM main.my_clean_table;
""").fetchone()[0]

print(f"✅ تم تحميل الملف بالكامل: {total:,} صف")


✅ تم تحميل الملف بالكامل: 10,000,000 صف


In [3]:
# فحص الرحلات التي تحتوي على معرّفات مكررة.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

duplicate_trip_ids_sample = """
SELECT trip_id, COUNT(*) 
FROM main.my_clean_table
WHERE trip_id IS NOT NULL AND trip_id != ''
GROUP BY trip_id
HAVING COUNT(*) > 1
LIMIT 10;
"""

duplicate_trip_ids_result = con.execute(duplicate_trip_ids_sample).df()

display(duplicate_trip_ids_result)



,trip_id,count_star()
0,TRIP_111742,2
1,TRIP_111877,2
2,TRIP_112029,2
3,TRIP_112042,2
4,TRIP_112186,2
5,TRIP_112734,2
6,TRIP_112956,2
7,TRIP_113012,2
8,TRIP_113028,2
9,TRIP_113135,2


In [4]:
# فحص الرحلات التي تحتوي على معرّف رحلة مفقود.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

missing_trip_id_sample = ("""
    SELECT COUNT(*)
    FROM main.my_clean_table
    WHERE trip_id IS NULL OR TRIM(trip_id) = ''
    LIMIT 10;
""")

missing_trip_id_result = con.execute(missing_trip_id_sample).df()

display(missing_trip_id_result)


,count_star()
0,49755


In [5]:
# إزالة معرّفات الرحلات غير الصالحة والتكرارات.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE VIEW main.v_step1_durations AS

SELECT *
FROM main.my_clean_table

WHERE trip_id IS NOT NULL
  AND TRIM(trip_id) != ''
  AND LOWER(TRIM(trip_id)) NOT IN ('null', 'nan', 'none')

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY TRIM(trip_id)
    ORDER BY start_time DESC NULLS LAST
) = 1;
""")

print("✅ Step 1: duplicate trip_id records resolved.")


✅ Step 1: duplicate trip_id records resolved.


In [6]:
# فحص الرحلات التي تحتوي على مدة زمنية غير صحيحة.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

invalid_duration_sample = ("""
SELECT trip_id, start_time, end_time
FROM main.v_step1_durations
WHERE end_time < start_time
LIMIT 10;
""")

invalid_duration_result = con.execute(invalid_duration_sample).df()

display(invalid_duration_result)



,trip_id,start_time,end_time
0,TRIP_1000051,2026-01-02 03:46:45,2026-01-02 03:04:45
1,TRIP_107052,2026-01-01 02:58:25,2026-01-01 02:31:25
2,TRIP_1087847,2026-01-02 06:13:04,2026-01-02 06:03:04
3,TRIP_1114235,2026-01-02 06:57:03,2026-01-02 06:37:03
4,TRIP_1133447,2026-01-02 07:29:04,2026-01-02 06:45:04
5,TRIP_1134401,2026-01-02 07:30:40,2026-01-02 06:50:40
6,TRIP_1157280,2026-01-02 08:08:48,2026-01-02 07:08:48
7,TRIP_1158027,2026-01-02 08:10:02,2026-01-02 08:00:02
8,TRIP_1200033,2026-01-02 09:20:03,2026-01-02 09:04:03
9,TRIP_1230623,2026-01-02 10:11:02,2026-01-02 09:34:02


In [7]:
# تنظيف أوقات الانتهاء غير الصحيحة وتحويلها إلى قيم فارغة.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE VIEW main.v_step2_durations AS

SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,

    CASE
        WHEN end_time >= start_time
        THEN end_time
        ELSE NULL
    END AS end_time,

    distance_km,
    fare_amount,
    status,
    phone_number,
    device_info,
    city

FROM main.v_step1_durations;
""")

print("✅ Step 2: invalid end_time values converted to NULL.")


✅ Step 2: invalid end_time values converted to NULL.


In [8]:
# فحص الرحلات التي يبدأ وقتها في المستقبل.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

future_trips_sample = con.execute("""
SELECT trip_id, start_time
FROM main.v_step2_durations
WHERE start_time > NOW()
LIMIT 10;
""").df()

display(future_trips_sample)
print("🔎 تم فحص الرحلات المستقبلية. ")


,trip_id,start_time
0,TRIP_5735604,2030-12-31
1,TRIP_5762727,2030-12-31
2,TRIP_5777948,2030-12-31
3,TRIP_5812987,2030-12-31
4,TRIP_5840859,2030-12-31
5,TRIP_5869864,2030-12-31
6,TRIP_5899718,2030-12-31
7,TRIP_5931911,2030-12-31
8,TRIP_5953549,2030-12-31
9,TRIP_5953988,2030-12-31


🔎 تم فحص الرحلات المستقبلية. 


In [9]:
# استبعاد الرحلات ذات أوقات البداية المستقبلية.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW  main.v_step3_future_trips AS
SELECT *
FROM main.v_step2_durations
WHERE start_time <= NOW();
""")
print("✅ تم استبعاد الرحلات المستقبلية. ")



future_trips_sample = con.execute(
    """
SELECT trip_id, start_time, end_time, city
FROM main.v_step3_future_trips
LIMIT 10;
"""
).df()

display(future_trips_sample)


✅ تم استبعاد الرحلات المستقبلية. 


,trip_id,start_time,end_time,city
0,TRIP_1000008,2026-01-02 03:46:40,2026-01-02 04:20:40,Khartoum
1,TRIP_1000051,2026-01-02 03:46:45,NaT,Port Sudan
2,TRIP_100014,2026-01-01 02:46:41,2026-01-01 03:05:41,Khartoum
3,TRIP_1000230,2026-01-02 03:47:03,2026-01-02 04:02:03,Khartoum
4,TRIP_1000283,2026-01-02 03:47:08,2026-01-02 04:06:08,Port Sudan
5,TRIP_1000303,2026-01-02 03:47:10,2026-01-02 04:28:10,Khartoum
6,TRIP_1000727,2026-01-02 03:47:52,2026-01-02 04:00:52,Khartoum
7,TRIP_1000826,2026-01-02 03:48:02,2026-01-02 04:12:02,Port Sudan
8,TRIP_1001047,2026-01-02 03:48:24,2026-01-02 04:24:24,Khartoum
9,TRIP_100127,2026-01-01 02:46:52,2026-01-01 03:11:52,Port Sudan


In [10]:
# فحص الرحلات المتداخلة للسائق نفسه في وقت البداية.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

overlapping_trips_sample = con.execute("""
SELECT driver_id, start_time, COUNT(*) 
FROM main.v_step3_future_trips
GROUP BY driver_id, start_time
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

display(overlapping_trips_sample)
print("🔎 تم فحص الرحلات المتداخلة. ")


,driver_id,start_time,count_star()
0,DRV_13,2026-01-07 14:32:51,2
1,DRV_222,2026-01-07 17:15:58,3
2,DRV_316,2026-01-07 17:36:53,2
3,DRV_123,2026-01-07 18:25:36,2
4,DRV_323,2026-01-07 18:35:08,2
5,DRV_635,2026-01-07 19:08:49,2
6,DRV_308,2026-01-07 19:12:58,2
7,DRV_676,2026-01-07 19:53:28,2
8,DRV_160,2026-01-07 21:06:35,2
9,DRV_506,2026-01-07 22:14:42,2


🔎 تم فحص الرحلات المتداخلة. 


In [11]:
# استبعاد الرحلات المتداخلة المكررة حسب السائق ووقت البداية.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW main.v_step4_overlapping_trips AS
SELECT * 
FROM main.v_step3_future_trips
QUALIFY ROW_NUMBER() OVER (PARTITION BY driver_id, start_time ORDER BY end_time DESC) = 1;
""")
print("✅ تم استبعاد الرحلات المتداخلة. ")



overlapping_trips_sample = con.execute("""
SELECT trip_id, start_time, end_time, driver_id
FROM main.v_step4_overlapping_trips
LIMIT 10;
"""
).df()

display(overlapping_trips_sample)


✅ تم استبعاد الرحلات المتداخلة. 


,trip_id,start_time,end_time,driver_id
0,TRIP_139711,2026-01-01 03:52:51,2026-01-01 04:29:51,DRV_1
1,TRIP_227300,2026-01-01 06:18:50,2026-01-01 07:03:50,DRV_1
2,TRIP_298411,2026-01-01 08:17:21,2026-01-01 09:00:21,DRV_1
3,TRIP_598565,2026-01-01 16:37:36,2026-01-01 17:15:36,DRV_1
4,TRIP_700929,2026-01-01 19:28:12,2026-01-01 20:12:12,DRV_1
5,TRIP_843437,2026-01-01 23:25:43,2026-01-02 00:07:43,DRV_1
6,TRIP_911485,2026-01-02 01:19:08,2026-01-02 02:03:08,DRV_1
7,TRIP_1048401,2026-01-02 05:07:20,2026-01-02 05:44:20,DRV_1
8,TRIP_1666643,2026-01-02 22:17:44,2026-01-02 22:44:44,DRV_1
9,TRIP_1732128,2026-01-03 00:06:52,2026-01-03 00:35:52,DRV_1


In [12]:
# فحص القيم السالبة في الأجرة والمسافة.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

negative_values_summary = con.execute("""
SELECT
    COUNT(CASE WHEN fare_amount < 0 THEN 1 END) AS negative_fares_count,
    COUNT(CASE WHEN distance_km < 0 THEN 1 END) AS negative_distances_count
FROM main.v_step4_overlapping_trips;
""").df()

display(negative_values_summary)

print("🔎 تم فحص القيم المالية والمسافات السالبة.")


,negative_fares_count,negative_distances_count
0,24607,97818


🔎 تم فحص القيم المالية والمسافات السالبة.


In [13]:
# إنشاء طبقة البيانات بعد إزالة التداخلات تمهيداً لتنظيف القيم الرقمية.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE VIEW main.v_step5_negative_values AS

SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,
    end_time,
    phone_number,
    device_info,
    city,
    status,
    distance_km,
    fare_amount

FROM main.v_step4_overlapping_trips;
""")

print(
    "✅ Step 5: negative values preserved for validation; "
    "no ABS() transformation applied."
)


✅ Step 5: negative values preserved for validation; no ABS() transformation applied.


In [14]:
# فحص القيم الشاذة في مبالغ الأجرة.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

fare_outliers_summary = con.execute("""
SELECT
    COUNT(CASE WHEN fare_amount > 5000 THEN 1 END) AS high_outliers_count,
    MAX(fare_amount) AS max_fare_in_data
FROM main.v_step5_negative_values
LIMIT 10;
""").df()

display(fare_outliers_summary)
print("🔎 تم فحص القيم الشاذة في الأجرة. ")


,high_outliers_count,max_fare_in_data
0,73262,1000000.0


🔎 تم فحص القيم الشاذة في الأجرة. 


In [15]:
# تنظيف القيم غير الصالحة في المسافة والأجرة مع الاحتفاظ بالقيم الشاذة للتحليل.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE VIEW main.v_step6_outliers_handled AS

SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,
    end_time,
    phone_number,
    device_info,
    city,
    status,

    CASE
        WHEN distance_km > 0
        THEN distance_km
        ELSE NULL
    END AS distance_km,

    CASE
        WHEN fare_amount >= 0
        THEN fare_amount
        ELSE NULL
    END AS fare_amount

FROM main.v_step5_negative_values;
""")

print(
    "✅ Step 6: invalid negative/zero distance and negative fares "
    "converted to NULL; no arbitrary fare cap applied."
)


✅ Step 6: invalid negative/zero distance and negative fares converted to NULL; no arbitrary fare cap applied.


In [16]:
# فحص جودة المسافة والقيم المفقودة أو الصفرية أو السالبة.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

distance_quality_summary = con.execute("""
SELECT
    COUNT(*) FILTER (WHERE distance_km < 0) AS negative_distance,
    COUNT(*) FILTER (WHERE distance_km = 0) AS zero_distance,
    COUNT(*) FILTER (WHERE distance_km IS NULL) AS missing_distance
FROM main.v_step5_negative_values;
""").df()

display(distance_quality_summary)

print("🔎 Distance quality profile completed.")


,negative_distance,zero_distance,missing_distance
0,97818,0,0


🔎 Distance quality profile completed.


In [17]:
# حساب السعر لكل كيلومتر مع تجنب القسمة على صفر.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW main.v_step7_division_handled AS
SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,
    end_time,
    phone_number,
    device_info,
    city,
    status,
    distance_km,
    fare_amount,

    fare_amount / NULLIF(distance_km, 0) AS price_per_km

FROM main.v_step6_outliers_handled;
""")
print("✅ تم إنشاء السعر لكل كيلومتر.")



df_TRAVEL = con.execute("""
SELECT *
FROM main.v_step7_division_handled
LIMIT 10;
"""
).df()

display(df_TRAVEL)


✅ تم إنشاء السعر لكل كيلومتر.


,trip_id,driver_id,customer_id,start_time,end_time,phone_number,device_info,city,status,distance_km,fare_amount,price_per_km
0,TRIP_139711,DRV_1,CST_1487,2026-01-01 03:52:51,2026-01-01 04:29:51,00249912345678,NULL,Port Sudan,failed,84.51,985.96,11.666785
1,TRIP_227300,DRV_1,CST_1483,2026-01-01 06:18:50,2026-01-01 07:03:50,09-123-4567,{},Omdurman,failed,423.69,161.87,0.382048
2,TRIP_298411,DRV_1,CST_402,2026-01-01 08:17:21,2026-01-01 09:00:21,N/A,"{""os"": ""iOS"", ""app_version"": ""5.1.0""}",Khartoum,NULL,244.46,1476.30,6.039025
3,TRIP_598565,DRV_1,CST_814,2026-01-01 16:37:36,2026-01-01 17:15:36,00249912345678,{},Khartoum,completed,13.69,194.19,14.184806
4,TRIP_700929,DRV_1,CST_1908,2026-01-01 19:28:12,2026-01-01 20:12:12,09-123-4567,{},Khartoum,NULL,85.31,312.13,3.658774
5,TRIP_843437,DRV_1,CST_1277,2026-01-01 23:25:43,2026-01-02 00:07:43,00249912345678,"{""os"": ""iOS"", ""app_version"": ""5.1.0""}",Bahri,Unknown,360.39,873.50,2.423763
6,TRIP_911485,DRV_1,CST_4119,2026-01-02 01:19:08,2026-01-02 02:03:08,09-123-4567,"{""os"": ""Android"", ""app_version"": ""5.1.2""}",Omdurman,COMPLETED,162.00,105.71,0.652531
7,TRIP_1048401,DRV_1,CST_155,2026-01-02 05:07:20,2026-01-02 05:44:20,00249912345678,NULL,Khartoum,Completed,115.16,50.82,0.441299
8,TRIP_1666643,DRV_1,CST_838,2026-01-02 22:17:44,2026-01-02 22:44:44,0912345678,"{""os"": ""Android"", ""app_version"": ""5.1.2""}",Bahri,completed,125.17,1221.76,9.760805
9,TRIP_1732128,DRV_1,CST_4399,2026-01-03 00:06:52,2026-01-03 00:35:52,09-123-4567,NULL,Bahri,Unknown,399.55,465.95,1.166187


In [18]:
# فحص جودة المدن وحالات النصوص.
import duckdb
con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

text_quality_summary = con.execute("""
SELECT city, COUNT(*) 
FROM main.v_step7_division_handled 
WHERE city != TRIM(city) 
GROUP BY city;

SELECT status, COUNT(*) 
FROM main.v_step7_division_handled 
GROUP BY status
LIMIT 10;
""").df()

display(text_quality_summary)
print("🔎 تم فحص جودة النصوص.")


,status,count_star()
0,Unknown,1219643
1,NaN,1218806
2,failed,1220906
3,NULL,1220823
4,completed,1218599
5,COMPLETED,1216653
6,Completed,1219103
7,Cancelled,1220954


🔎 تم فحص جودة النصوص.


In [19]:
# تنظيف المدن وحالات النصوص وتوحيد القيم.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW main.v_step8_text_cleaned AS
SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,
    end_time,
    phone_number,
    device_info,
    distance_km,
    fare_amount,
    price_per_km,

    TRIM(city) AS city,

    CASE
        WHEN TRIM(UPPER(status)) IN
            ('NULL', 'NAN', 'UNKNOWN', 'N/A', '')
        THEN NULL
        ELSE TRIM(UPPER(status))
    END AS status

FROM main.v_step7_division_handled;
""")

print("✅ Step 8: تم تنظيف النصوص والحالات بنجاح.")


✅ Step 8: تم تنظيف النصوص والحالات بنجاح.


In [20]:
# فحص المدن التي تحتوي على رموز غير صالحة.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

invalid_city_chars = con.execute("""
SELECT city, COUNT(*) as count
FROM main.v_step8_text_cleaned
WHERE REGEXP_MATCHES(city, '[^a-zA-Z\\s]')
GROUP BY city;
""").df()

display(invalid_city_chars)


,city,count


In [21]:
# فحص تنسيق أرقام الهواتف قبل تنظيفها.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

phone_format_summary = con.execute("""
SELECT phone_number, COUNT(*) AS count
FROM main.v_step8_text_cleaned
GROUP BY phone_number
LIMIT 10;
""").df()

display(phone_format_summary)


,phone_number,count
0,09-123-4567,1950991
1,+249912345678,1951026
2,00249912345678,1950098
3,N/A,1951573
4,0912345678,1951799


In [22]:
# تنظيف المدن من الرموز غير المنطقية.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW main.v_step9_clean_chars AS
SELECT
    *,
    TRIM(
        REGEXP_REPLACE(
            city,
            '[^a-zA-Z ]',
            '',
            'g'
        )
    ) AS city_clean
FROM main.v_step8_text_cleaned;
""")

invalid_city_chars = con.execute("""
SELECT
    city,
    COUNT(*) AS count
FROM main.v_step8_text_cleaned
WHERE REGEXP_MATCHES(city, '[^a-zA-Z ]')
GROUP BY city;
""").df()

display(invalid_city_chars)

print("✅ Step 9: تم تنظيف الرموز غير المنطقية من المدن.")


,city,count


✅ Step 9: تم تنظيف الرموز غير المنطقية من المدن.


In [23]:
# تنظيف أرقام الهواتف وتوحيد صيغة الأرقام الصالحة.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE VIEW main.v_step9_clean_phones AS

SELECT
    * EXCLUDE (phone_number, city),

    CASE

        WHEN phone_number IS NULL
          OR LOWER(TRIM(phone_number))
                IN ('', 'null', 'nan', 'none', 'n/a')
        THEN NULL

        WHEN REGEXP_REPLACE(
                TRIM(phone_number),
                '[^0-9]',
                '',
                'g'
             ) LIKE '00249%'
             AND LENGTH(
                REGEXP_REPLACE(
                    TRIM(phone_number),
                    '[^0-9]',
                    '',
                    'g'
                )
             ) = 14
        THEN
            '+249' ||
            SUBSTR(
                REGEXP_REPLACE(
                    TRIM(phone_number),
                    '[^0-9]',
                    '',
                    'g'
                ),
                6
            )

        WHEN REGEXP_REPLACE(
                TRIM(phone_number),
                '[^0-9]',
                '',
                'g'
             ) LIKE '249%'
             AND LENGTH(
                REGEXP_REPLACE(
                    TRIM(phone_number),
                    '[^0-9]',
                    '',
                    'g'
                )
             ) = 12
        THEN
            '+' ||
            REGEXP_REPLACE(
                TRIM(phone_number),
                '[^0-9]',
                '',
                'g'
            )

        WHEN REGEXP_REPLACE(
                TRIM(phone_number),
                '[^0-9]',
                '',
                'g'
             ) LIKE '09%'
             AND LENGTH(
                REGEXP_REPLACE(
                    TRIM(phone_number),
                    '[^0-9]',
                    '',
                    'g'
                )
             ) = 10
        THEN
            '+249' ||
            SUBSTR(
                REGEXP_REPLACE(
                    TRIM(phone_number),
                    '[^0-9]',
                    '',
                    'g'
                ),
                2
            )

        ELSE NULL

    END AS clean_phone,

    city_clean AS city

FROM main.v_step9_clean_chars;
""")

print("✅ Step 9: phones and city cleaning connected to the main pipeline.")


✅ Step 9: phones and city cleaning connected to the main pipeline.


In [24]:
# فحص سجلات معلومات الأجهزة التي لا تحتوي على JSON صالح.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

invalid_json_summary = con.execute("""
SELECT device_info, COUNT(*) as count
FROM main.v_step9_clean_phones
WHERE device_info IS NOT NULL 
  AND device_info != ''
  AND json_valid(device_info) = FALSE -- اعزل فقط غير الصالح هندسياً
GROUP BY device_info;
""").df()

display(invalid_json_summary)


,device_info,count
0,NULL,1950366


In [25]:
# التحقق من JSON واستخراج نظام التشغيل وإصدار التطبيق.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

con.execute("""
CREATE OR REPLACE VIEW main.v_step10_parsed_json AS

WITH validated_json AS (

    SELECT
        *,

        CASE
            WHEN device_info IS NOT NULL
                 AND TRIM(device_info) != ''
                 AND LOWER(TRIM(device_info))
                     NOT IN ('null', 'nan', 'none')
                 AND json_valid(device_info)
            THEN device_info

            ELSE NULL
        END AS valid_device_json

    FROM main.v_step9_clean_phones
)

SELECT
    *,

    json_extract_string(
        valid_device_json,
        '$.os'
    ) AS device_os,

    json_extract_string(
        valid_device_json,
        '$.app_version'
    ) AS device_app_version

FROM validated_json;
""")

device_info_summary = con.execute("""
SELECT
    trip_id,
    device_info,
    device_os,
    device_app_version
FROM main.v_step10_parsed_json
LIMIT 10;
""").df()

display(device_info_summary)

print("✅ Step 10: تم فحص JSON واستخراج البيانات بأمان.")


,trip_id,device_info,device_os,device_app_version
0,TRIP_139711,NULL,None,None
1,TRIP_227300,{},None,None
2,TRIP_298411,"{""os"": ""iOS"", ""app_version"": ""5.1.0""}",iOS,5.1.0
3,TRIP_598565,{},None,None
4,TRIP_700929,{},None,None
5,TRIP_843437,"{""os"": ""iOS"", ""app_version"": ""5.1.0""}",iOS,5.1.0
6,TRIP_911485,"{""os"": ""Android"", ""app_version"": ""5.1.2""}",Android,5.1.2
7,TRIP_1048401,NULL,None,None
8,TRIP_1666643,"{""os"": ""Android"", ""app_version"": ""5.1.2""}",Android,5.1.2
9,TRIP_1732128,NULL,None,None


✅ Step 10: تم فحص JSON واستخراج البيانات بأمان.


In [26]:
# فحص مراجع السائقين والعملاء غير الموجودة في البيانات المرجعية.
import duckdb

con = duckdb.connect(database="/tmp/enterprise_warehouse.duckdb")

orphan_reference_sample = con.execute("""
SELECT trip_id, driver_id, customer_id
FROM main.v_step10_parsed_json
WHERE 
   driver_id IS NULL 
   OR TRIM(driver_id) = '' 
   OR LOWER(TRIM(driver_id)) IN ('null', 'nan', 'none')
   
   OR customer_id IS NULL 
   OR TRIM(customer_id) = '' 
   OR LOWER(TRIM(customer_id)) IN ('null', 'nan', 'none');
""").df()

print(f"✅ عدد السجلات اليتيمة المكتشفة بدقة: {len(orphan_reference_sample)}")
display(orphan_reference_sample)


✅ عدد السجلات اليتيمة المكتشفة بدقة: 0


,trip_id,driver_id,customer_id


In [27]:
# إنشاء جداول المراجع للسائقين والعملاء والمواقع.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE TABLE main.ref_drivers AS
SELECT
    'DRV_' || CAST(i AS VARCHAR) AS driver_id
FROM range(1, 1001) AS t(i);
""")

con.execute("""
CREATE OR REPLACE TABLE main.ref_customers AS
SELECT
    'CST_' || CAST(i AS VARCHAR) AS customer_id
FROM range(1, 5001) AS t(i);
""")

print("✅ Reference tables created.")

display(
    con.execute("""
        SELECT
            'drivers' AS reference_name,
            COUNT(*) AS rows_count
        FROM main.ref_drivers

        UNION ALL

        SELECT
            'customers',
            COUNT(*)
        FROM main.ref_customers;
    """).df()
)


✅ Reference tables created.


,reference_name,rows_count
0,drivers,1000
1,customers,5000


In [28]:
# إنشاء جدول الرحلات النهائي من البيانات المنظفة.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE TABLE main.fct_trips AS

WITH final_clean AS (

    SELECT
        trip_id,
        driver_id,
        customer_id,
        start_time,
        end_time,
        distance_km,
        fare_amount,
        price_per_km,

        TRIM(
            REGEXP_REPLACE(
                city,
                '[^a-zA-Z ]',
                '',
                'g'
            )
        ) AS city_clean,

        status,
        clean_phone,
        device_os,
        device_app_version

    FROM main.v_step10_parsed_json
)

SELECT
    trip_id,
    driver_id,
    customer_id,
    start_time,
    end_time,
    distance_km,
    fare_amount,
    price_per_km,
    city_clean AS city,
    status,
    clean_phone AS phone_number,
    device_os,
    device_app_version

FROM final_clean

WHERE
    trip_id IS NOT NULL
    AND TRIM(trip_id) != ''
    AND LOWER(TRIM(trip_id))
        NOT IN ('null', 'nan', 'none')

    AND driver_id IS NOT NULL
    AND TRIM(driver_id) != ''

    AND customer_id IS NOT NULL
    AND TRIM(customer_id) != ''

    AND start_time IS NOT NULL

    AND end_time IS NOT NULL
    AND end_time >= start_time

    AND distance_km IS NOT NULL
    AND distance_km > 0

    AND fare_amount IS NOT NULL
    AND fare_amount >= 0

    AND city_clean IS NOT NULL
    AND TRIM(city_clean) != '';
""")

final_rows = con.execute("""
SELECT COUNT(*)
FROM main.fct_trips;
""").fetchone()[0]

print(
    f"✅ fct_trips created successfully: {final_rows:,} rows."
)

display(
    con.execute("""
        SELECT *
        FROM main.fct_trips
        LIMIT 10;
    """).df()
)


✅ fct_trips created successfully: 9,585,012 rows.


,trip_id,driver_id,customer_id,start_time,end_time,distance_km,fare_amount,price_per_km,city,status,phone_number,device_os,device_app_version
0,TRIP_139711,DRV_1,CST_1487,2026-01-01 03:52:51,2026-01-01 04:29:51,84.51,985.96,11.666785,Port Sudan,FAILED,+249912345678,None,None
1,TRIP_227300,DRV_1,CST_1483,2026-01-01 06:18:50,2026-01-01 07:03:50,423.69,161.87,0.382048,Omdurman,FAILED,None,None,None
2,TRIP_298411,DRV_1,CST_402,2026-01-01 08:17:21,2026-01-01 09:00:21,244.46,1476.30,6.039025,Khartoum,None,None,iOS,5.1.0
3,TRIP_598565,DRV_1,CST_814,2026-01-01 16:37:36,2026-01-01 17:15:36,13.69,194.19,14.184806,Khartoum,COMPLETED,+249912345678,None,None
4,TRIP_700929,DRV_1,CST_1908,2026-01-01 19:28:12,2026-01-01 20:12:12,85.31,312.13,3.658774,Khartoum,None,None,None,None
5,TRIP_843437,DRV_1,CST_1277,2026-01-01 23:25:43,2026-01-02 00:07:43,360.39,873.50,2.423763,Bahri,None,+249912345678,iOS,5.1.0
6,TRIP_911485,DRV_1,CST_4119,2026-01-02 01:19:08,2026-01-02 02:03:08,162.00,105.71,0.652531,Omdurman,COMPLETED,None,Android,5.1.2
7,TRIP_1048401,DRV_1,CST_155,2026-01-02 05:07:20,2026-01-02 05:44:20,115.16,50.82,0.441299,Khartoum,COMPLETED,+249912345678,None,None
8,TRIP_1666643,DRV_1,CST_838,2026-01-02 22:17:44,2026-01-02 22:44:44,125.17,1221.76,9.760805,Bahri,COMPLETED,+249912345678,Android,5.1.2
9,TRIP_1732128,DRV_1,CST_4399,2026-01-03 00:06:52,2026-01-03 00:35:52,399.55,465.95,1.166187,Bahri,None,None,None,None


In [29]:
# التحقق من مراجع السائقين والعملاء وإنشاء الجدول النهائي المعتمد.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE TABLE main.fct_trips_final AS

SELECT
    f.*

FROM main.fct_trips f

INNER JOIN main.ref_drivers d
    ON TRIM(f.driver_id) = TRIM(d.driver_id)

INNER JOIN main.ref_customers c
    ON TRIM(f.customer_id) = TRIM(c.customer_id)

WHERE


    f.trip_id IS NOT NULL
    AND TRIM(f.trip_id) != ''


    AND f.driver_id IS NOT NULL
    AND TRIM(f.driver_id) != ''

    AND f.customer_id IS NOT NULL
    AND TRIM(f.customer_id) != ''


    AND f.start_time IS NOT NULL
    AND f.start_time <= NOW()

    AND f.end_time IS NOT NULL
    AND f.end_time >= f.start_time


    AND f.distance_km IS NOT NULL
    AND f.distance_km > 0


    AND f.fare_amount IS NOT NULL
    AND f.fare_amount >= 0


    AND f.status IS NOT NULL
    AND LOWER(TRIM(f.status))
        IN (
            'completed',
            'cancelled',
            'failed'
        )


    AND f.city IS NOT NULL
    AND LOWER(TRIM(f.city))
        IN (
            'khartoum',
            'omdurman',
            'bahri',
            'port sudan'
        )

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY TRIM(f.trip_id)
    ORDER BY f.start_time DESC NULLS LAST
) = 1;
""")

final_stats = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT trip_id) AS unique_trip_ids
FROM main.fct_trips_final;
""").df()

display(final_stats)

print("✅ fct_trips_final created successfully.")


,total_rows,unique_trip_ids
0,5931861,5931861


✅ fct_trips_final created successfully.


In [30]:
# إنشاء اختبارات جودة البيانات وتسجيل نتائجها.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

con.execute("""
CREATE OR REPLACE TABLE main.test_results (
    test_name VARCHAR,
    expected VARCHAR,
    actual VARCHAR,
    status VARCHAR,
    run_time TIMESTAMP
);
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Uniqueness Check (trip_id)',
    '0 duplicates',
    CAST(COUNT(*) AS VARCHAR) || ' duplicates',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM (
    SELECT trip_id
    FROM main.fct_trips_final
    GROUP BY trip_id
    HAVING COUNT(*) > 1
);
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Required IDs Check',
    '0 invalid rows',
    CAST(COUNT(*) AS VARCHAR) || ' invalid rows',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE trip_id IS NULL OR TRIM(trip_id) = ''
   OR driver_id IS NULL OR TRIM(driver_id) = ''
   OR customer_id IS NULL OR TRIM(customer_id) = '';
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Driver Reference Integrity',
    '0 invalid references',
    CAST(COUNT(*) AS VARCHAR) || ' invalid references',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final f
LEFT JOIN main.ref_drivers d
    ON TRIM(f.driver_id) = TRIM(d.driver_id)
WHERE d.driver_id IS NULL;
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Customer Reference Integrity',
    '0 invalid references',
    CAST(COUNT(*) AS VARCHAR) || ' invalid references',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final f
LEFT JOIN main.ref_customers c
    ON TRIM(f.customer_id) = TRIM(c.customer_id)
WHERE c.customer_id IS NULL;
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'invalid_duration_sample Check',
    '0 invalid intervals',
    CAST(COUNT(*) AS VARCHAR) || ' invalid intervals',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE start_time IS NULL
   OR end_time IS NULL
   OR end_time < start_time
   OR start_time > NOW();
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Distance Check',
    '0 invalid values',
    CAST(COUNT(*) AS VARCHAR) || ' invalid values',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE distance_km IS NULL
   OR distance_km <= 0;
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Fare Check',
    '0 invalid values',
    CAST(COUNT(*) AS VARCHAR) || ' invalid values',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE fare_amount IS NULL
   OR fare_amount < 0;
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Status Check',
    '0 invalid statuses',
    CAST(COUNT(*) AS VARCHAR) || ' invalid statuses',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE status IS NULL
   OR LOWER(TRIM(status))
        NOT IN ('completed', 'cancelled', 'failed');
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'City Check',
    '0 invalid cities',
    CAST(COUNT(*) AS VARCHAR) || ' invalid cities',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE city IS NULL
   OR LOWER(TRIM(city))
        NOT IN ('khartoum', 'omdurman', 'bahri', 'port sudan');
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'Phone Format Check',
    '0 invalid phones',
    CAST(COUNT(*) AS VARCHAR) || ' invalid phones',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.fct_trips_final
WHERE phone_number IS NOT NULL
  AND NOT REGEXP_MATCHES(
      phone_number,
      '^\\+2499[0-9]{8}$'
  );
""")


con.execute("""
INSERT INTO main.test_results
SELECT
    'JSON Validation Check',
    '0 invalid JSON values',
    CAST(COUNT(*) AS VARCHAR) || ' invalid JSON values',
    CASE
        WHEN COUNT(*) = 0 THEN 'PASS'
        ELSE 'FAIL'
    END,
    CURRENT_TIMESTAMP
FROM main.v_step10_parsed_json
WHERE device_info IS NOT NULL
  AND TRIM(device_info) != ''
  AND LOWER(TRIM(device_info))
        NOT IN ('null', 'nan', 'none')
  AND json_valid(device_info) = FALSE;
""")

print("✅ Final data-quality tests executed.")

display(
    con.execute("""
        SELECT
            test_name,
            expected,
            actual,
            status,
            run_time
        FROM main.test_results
        ORDER BY test_name;
    """).df()
)


✅ Final data-quality tests executed.


,test_name,expected,actual,status,run_time
0,City Check,0 invalid cities,0 invalid cities,PASS,2026-09-02 07:15:37.958625
1,Customer Reference Integrity,0 invalid references,0 invalid references,PASS,2026-09-02 07:15:37.459115
2,Distance Check,0 invalid values,0 invalid values,PASS,2026-09-02 07:15:37.874867
3,Driver Reference Integrity,0 invalid references,0 invalid references,PASS,2026-09-02 07:15:37.406470
4,Fare Check,0 invalid values,0 invalid values,PASS,2026-09-02 07:15:37.881960
5,JSON Validation Check,0 invalid JSON values,0 invalid JSON values,PASS,2026-09-02 07:15:38.210036
6,Phone Format Check,0 invalid phones,0 invalid phones,PASS,2026-09-02 07:15:38.037659
7,Required IDs Check,0 invalid rows,0 invalid rows,PASS,2026-09-02 07:15:37.081703
8,Status Check,0 invalid statuses,0 invalid statuses,PASS,2026-09-02 07:15:37.883807
9,Uniqueness Check (trip_id),0 duplicates,0 duplicates,PASS,2026-09-02 07:15:36.436282


In [31]:
# تشغيل بوابة جودة البيانات وإيقاف خط الأنابيب عند وجود فشل.

failed_tests = con.execute("""
    SELECT COUNT(*)
    FROM main.test_results
    WHERE status = 'FAIL';
""").fetchone()[0]

if failed_tests > 0:

    print(f"❌ Pipeline متوقف: {failed_tests} اختبار فشل.\n")

    print("الاختبارات الفاشلة:")
    
    display(
        con.execute("""
            SELECT
                test_name,
                expected,
                actual,
                status,
                run_time
            FROM main.test_results
            WHERE status = 'FAIL'
            ORDER BY test_name;
        """).df()
    )

    raise RuntimeError(
        f"Pipeline متوقف بسبب {failed_tests} اختبار فاشل. "
        "راجع نتائج اختبارات الجودة أعلاه."
    )

else:

    print(
        "✅ جميع اختبارات الجودة PASS. "
        "البيانات جاهزة للطبقة التحليلية."
    )


✅ جميع اختبارات الجودة PASS. البيانات جاهزة للطبقة التحليلية.


In [32]:
# إنشاء جداول طبقة الأعمال للعملاء والسائقين والإيرادات.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)



con.execute("""
CREATE OR REPLACE TABLE main.biz_customer_segmentation AS

SELECT
    customer_id,

    ANY_VALUE(phone_number) AS phone_number,

    COUNT(*) AS total_trips,

    COUNT(*) FILTER (
        WHERE status = 'COMPLETED'
    ) AS completed_trips,

    COUNT(*) FILTER (
        WHERE status = 'CANCELLED'
    ) AS cancelled_trips,

    COUNT(*) FILTER (
        WHERE status = 'FAILED'
    ) AS failed_trips,

    ROUND(
        SUM(fare_amount),
        2
    ) AS total_spend,

    ROUND(
        AVG(fare_amount),
        2
    ) AS average_fare,

    ROUND(
        AVG(distance_km),
        2
    ) AS average_distance,

    CASE
        WHEN SUM(fare_amount) >= 50000
          OR COUNT(*) >= 50
        THEN 'VIP'

        WHEN SUM(fare_amount) >= 10000
        THEN 'REGULAR'

        ELSE 'LOW_FREQUENT'
    END AS customer_tier

FROM main.fct_trips_final

GROUP BY customer_id;
""")


con.execute("""
CREATE OR REPLACE TABLE main.biz_driver_performance AS

SELECT
    driver_id,

    COUNT(*) AS total_trips,

    COUNT(*) FILTER (
        WHERE status = 'COMPLETED'
    ) AS completed_trips,

    COUNT(*) FILTER (
        WHERE status = 'CANCELLED'
    ) AS cancelled_trips,

    COUNT(*) FILTER (
        WHERE status = 'FAILED'
    ) AS failed_trips,

    ROUND(
        SUM(
            CASE
                WHEN status = 'COMPLETED'
                THEN fare_amount
                ELSE 0
            END
        ),
        2
    ) AS total_earnings,

    ROUND(
        AVG(
            CASE
                WHEN status = 'COMPLETED'
                THEN distance_km
            END
        ),
        2
    ) AS avg_trip_distance,

    ROUND(
        AVG(
            CASE
                WHEN status = 'COMPLETED'
                THEN fare_amount
            END
        ),
        2
    ) AS avg_completed_fare

FROM main.fct_trips_final

GROUP BY driver_id;
""")



con.execute("""
CREATE OR REPLACE TABLE main.dim_customer AS

SELECT
    customer_id,
    phone_number,
    total_trips,
    completed_trips,
    cancelled_trips,
    failed_trips,
    total_spend,
    average_fare,
    average_distance,
    customer_tier

FROM main.biz_customer_segmentation;
""")


con.execute("""
CREATE OR REPLACE TABLE main.dim_driver AS

SELECT
    driver_id,
    total_trips,
    completed_trips,
    cancelled_trips,
    failed_trips,
    total_earnings,
    avg_trip_distance,
    avg_completed_fare

FROM main.biz_driver_performance;
""")


con.execute("""
CREATE OR REPLACE TABLE main.dim_location AS

SELECT
    city,

    COUNT(*) AS total_trips,

    COUNT(*) FILTER (
        WHERE status = 'COMPLETED'
    ) AS completed_trips,

    COUNT(*) FILTER (
        WHERE status = 'CANCELLED'
    ) AS cancelled_trips,

    COUNT(*) FILTER (
        WHERE status = 'FAILED'
    ) AS failed_trips,

    ROUND(
        SUM(fare_amount),
        2
    ) AS total_revenue,

    ROUND(
        AVG(fare_amount),
        2
    ) AS average_fare,

    ROUND(
        AVG(distance_km),
        2
    ) AS average_distance

FROM main.fct_trips_final

GROUP BY city;
""")


con.execute("""
CREATE OR REPLACE TABLE main.fact_daily_revenue AS

SELECT
    CAST(start_time AS DATE) AS trip_date,

    city,

    COUNT(*) AS daily_trips,

    COUNT(*) FILTER (
        WHERE status = 'COMPLETED'
    ) AS completed_trips,

    COUNT(*) FILTER (
        WHERE status = 'CANCELLED'
    ) AS cancelled_trips,

    COUNT(*) FILTER (
        WHERE status = 'FAILED'
    ) AS failed_trips,

    ROUND(
        SUM(fare_amount),
        2
    ) AS daily_revenue,

    ROUND(
        AVG(fare_amount),
        2
    ) AS average_fare

FROM main.fct_trips_final

GROUP BY
    CAST(start_time AS DATE),
    city;
""")

print("✅ Business Layer + Star Schema created successfully.")

display(
    con.execute("""
        SELECT
            'biz_customer_segmentation' AS table_name,
            COUNT(*) AS rows_count
        FROM main.biz_customer_segmentation

        UNION ALL

        SELECT
            'biz_driver_performance',
            COUNT(*)
        FROM main.biz_driver_performance

        UNION ALL

        SELECT
            'dim_customer',
            COUNT(*)
        FROM main.dim_customer

        UNION ALL

        SELECT
            'dim_driver',
            COUNT(*)
        FROM main.dim_driver

        UNION ALL

        SELECT
            'dim_location',
            COUNT(*)
        FROM main.dim_location

        UNION ALL

        SELECT
            'fact_daily_revenue',
            COUNT(*)
        FROM main.fact_daily_revenue;
    """).df()
)


✅ Business Layer + Star Schema created successfully.


,table_name,rows_count
0,biz_customer_segmentation,5000
1,biz_driver_performance,1000
2,dim_customer,5000
3,dim_driver,1000
4,dim_location,4
5,fact_daily_revenue,48


In [33]:
# إنشاء التحليلات المشتقة لترتيب السائقين ونمو الإيرادات اليومي.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)


con.execute("""
CREATE OR REPLACE TABLE main.analytical_driver_insights AS

SELECT
    driver_id,
    completed_trips,
    total_earnings,

    RANK() OVER (
        ORDER BY total_earnings DESC
    ) AS earnings_rank,

    NTILE(4) OVER (
        ORDER BY completed_trips DESC
    ) AS activity_quartile

FROM main.biz_driver_performance;
""")


con.execute("""
CREATE OR REPLACE TABLE main.analytical_daily_growth AS

WITH daily AS (

    SELECT
        city,
        trip_date,
        daily_revenue,

        LAG(daily_revenue) OVER (
            PARTITION BY city
            ORDER BY trip_date
        ) AS previous_day_revenue

    FROM main.fact_daily_revenue
)

SELECT
    city,
    trip_date,
    daily_revenue,
    previous_day_revenue,

    CASE
        WHEN previous_day_revenue IS NULL
             OR previous_day_revenue = 0
        THEN NULL

        ELSE ROUND(
            (
                daily_revenue - previous_day_revenue
            )
            / previous_day_revenue
            * 100,
            2
        )
    END AS growth_percentage

FROM daily;
""")

print("✅ Advanced analytical tables created successfully.")

display(
    con.execute("""
        SELECT
            'analytical_driver_insights' AS table_name,
            COUNT(*) AS rows_count
        FROM main.analytical_driver_insights

        UNION ALL

        SELECT
            'analytical_daily_growth',
            COUNT(*)
        FROM main.analytical_daily_growth;
    """).df()
)


✅ Advanced analytical tables created successfully.


,table_name,rows_count
0,analytical_driver_insights,1000
1,analytical_daily_growth,48


In [34]:
# مقارنة أعداد السجلات للتحقق من اتساق الجداول النهائية.
import duckdb

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)

summary = con.execute("""
SELECT
    'fct_trips_final' AS table_name,
    COUNT(*) AS rows_count
FROM main.fct_trips_final

UNION ALL

SELECT
    'biz_customer_segmentation',
    COUNT(*)
FROM main.biz_customer_segmentation

UNION ALL

SELECT
    'biz_driver_performance',
    COUNT(*)
FROM main.biz_driver_performance

UNION ALL

SELECT
    'dim_customer',
    COUNT(*)
FROM main.dim_customer

UNION ALL

SELECT
    'dim_driver',
    COUNT(*)
FROM main.dim_driver

UNION ALL

SELECT
    'dim_location',
    COUNT(*)
FROM main.dim_location

UNION ALL

SELECT
    'fact_daily_revenue',
    COUNT(*)
FROM main.fact_daily_revenue

UNION ALL

SELECT
    'analytical_driver_insights',
    COUNT(*)
FROM main.analytical_driver_insights

UNION ALL

SELECT
    'analytical_daily_growth',
    COUNT(*)
FROM main.analytical_daily_growth;
""").df()

display(summary)

print("✅ Final reconciliation completed.")


,table_name,rows_count
0,fct_trips_final,5931861
1,biz_customer_segmentation,5000
2,biz_driver_performance,1000
3,dim_customer,5000
4,dim_driver,1000
5,dim_location,4
6,fact_daily_revenue,48
7,analytical_driver_insights,1000
8,analytical_daily_growth,48


✅ Final reconciliation completed.


In [35]:
# تصدير الجداول النهائية إلى ملفات CSV وParquet.
import duckdb
from pathlib import Path

con = duckdb.connect(
    database="/tmp/enterprise_warehouse.duckdb"
)


export_dir = Path("final_exports")
export_dir.mkdir(
    parents=True,
    exist_ok=True
)

exports = {
    "final_clean_trips.csv": "fct_trips_final",
    "customers.csv": "dim_customer",
    "drivers.csv": "dim_driver",
    "locations.csv": "dim_location",
    "daily_revenue.csv": "fact_daily_revenue",
    "driver_insights.csv": "analytical_driver_insights",
    "daily_growth.csv": "analytical_daily_growth",

    "final_clean_trips.parquet": "fct_trips_final",
    "customers.parquet": "dim_customer",
    "drivers.parquet": "dim_driver",
    "locations.parquet": "dim_location",
    "daily_revenue.parquet": "fact_daily_revenue",
    "driver_insights.parquet": "analytical_driver_insights",
    "daily_growth.parquet": "analytical_daily_growth"
}

for filename, table_name in exports.items():

    output_path = export_dir / filename

    if output_path.exists():
        output_path.unlink()

    if filename.endswith(".csv"):

        con.execute(f"""
            COPY main."{table_name}"
            TO '{output_path.as_posix()}'
            (
                HEADER,
                DELIMITER ','
            );
        """)

    else:

        con.execute(f"""
            COPY main."{table_name}"
            TO '{output_path.as_posix()}'
            (
                FORMAT PARQUET,
                COMPRESSION 'ZSTD'
            );
        """)

    print(f"✅ Exported: {output_path}")

print("✅ ALL FINAL EXPORTS COMPLETED.")


✅ Exported: final_exports/final_clean_trips.csv
✅ Exported: final_exports/customers.csv
✅ Exported: final_exports/drivers.csv
✅ Exported: final_exports/locations.csv
✅ Exported: final_exports/daily_revenue.csv
✅ Exported: final_exports/driver_insights.csv
✅ Exported: final_exports/daily_growth.csv
✅ Exported: final_exports/final_clean_trips.parquet
✅ Exported: final_exports/customers.parquet
✅ Exported: final_exports/drivers.parquet
✅ Exported: final_exports/locations.parquet
✅ Exported: final_exports/daily_revenue.parquet
✅ Exported: final_exports/driver_insights.parquet
✅ Exported: final_exports/daily_growth.parquet
✅ ALL FINAL EXPORTS COMPLETED.


In [36]:
# تسجيل تشغيل خط الأنابيب وعدد السجلات المعالجة.
con.execute("""
CREATE TABLE IF NOT EXISTS main.pipeline_execution_logs (
    execution_id BIGINT,
    pipeline_name VARCHAR,
    run_date TIMESTAMP,
    rows_processed BIGINT,
    status VARCHAR
);
""")

total_rows_processed = con.execute("""
SELECT COUNT(*)
FROM main.fct_trips_final;
""").fetchone()[0]

next_execution_id = con.execute("""
SELECT
    COALESCE(MAX(execution_id), 0) + 1
FROM main.pipeline_execution_logs;
""").fetchone()[0]

con.execute("""
INSERT INTO main.pipeline_execution_logs
VALUES (?, ?, CURRENT_TIMESTAMP, ?, ?);
""", [
    next_execution_id,
    'Enterprise_Trips_Pipeline_FINAL',
    total_rows_processed,
    'SUCCESS'
])

print("✅ تم تسجيل تشغيل الـPipeline.")

display(
    con.execute("""
        SELECT *
        FROM main.pipeline_execution_logs
        ORDER BY execution_id DESC
        LIMIT 5;
    """).df()
)


✅ تم تسجيل تشغيل الـPipeline.


,execution_id,pipeline_name,run_date,rows_processed,status
0,1,Enterprise_Trips_Pipeline_FINAL,2026-09-02 07:16:14.285514,5931861,SUCCESS


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=66524818-c70c-4628-bdd3-72f6d975670b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>